# 05 — Performance Evaluation
**FootScout — BHT Berlin**

Mandatory WP: Precision@k, Recall@k, NDCG@k — two evaluation methods.


In [ ]:
import sys; sys.path.insert(0,"..")
import pandas as pd, wandb
from src.embeddings import load_embeddings
from src.recommender import FootScoutRecommender
from src.evaluate import (run_comparison_experiment, PositionBasedEvaluator,
                           TransfermarktBenchmarkEvaluator)
df = pd.read_csv("../data/processed/players_merged.csv", low_memory=False)
embs = load_embeddings(method="umap", alpha=0.6)

## Method 1 — Position-Based Ground Truth

In [ ]:
rec = FootScoutRecommender(df=df, embeddings=embs["hybrid"])
pos_eval = PositionBasedEvaluator(df)
m1 = pos_eval.evaluate(rec, k_values=[3,5,10], n_queries=50)
print(pos_eval.summarize(m1).to_string())

## Method 2 — Transfermarkt Benchmark

In [ ]:
tm_eval = TransfermarktBenchmarkEvaluator()
m2 = tm_eval.evaluate(rec, k_values=[3,5,10])
if not m2.empty:
    print(m2.groupby("k")[["precision_at_k","recall_at_k","ndcg_at_k"]].mean().round(4))

## Full Comparison: stat vs text vs hybrid

In [ ]:
run = wandb.init(project="footscout", name="eval_comparison", job_type="eval")
cmp = run_comparison_experiment(df, {"stat":embs["stat"],"text":embs["text"],"hybrid":embs["hybrid"]},
    k_values=[3,5,10], n_queries=50, wandb_run=run)
print(cmp.groupby(["embedding_type","k"])[["precision_at_k","recall_at_k","ndcg_at_k"]].mean().round(4))
run.finish()

## NDCG@k Comparison Chart

In [ ]:
import plotly.express as px
s = cmp.groupby(["embedding_type","k"])["ndcg_at_k"].mean().reset_index()
fig = px.line(s, x="k", y="ndcg_at_k", color="embedding_type",
    markers=True, title="NDCG@k by Embedding Type", template="plotly_dark")
fig.show()